# Integração técnica do SAM 1 ViT-B

Hoje verificaremos se o SAM consegue receber prompts produzidos somente pela DR-IFT. Este é um teste técnico em quatro casos fixos da validação; ele não escolhe o melhor modo e não acessa teste ou FUSeg.

Fluxo: **imagem → DR-IFT → componentes/box/logits → SAM → nova máscara**.


## 1. Preparar o Colab

A cópia do Dataset para o disco temporário acelera a leitura. O projeto e os resultados permanecem no Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, urllib.request, yaml

PROJECT_ROOT = Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
DATA_ROOT = Path('/content/tcc_data')
%cd {PROJECT_ROOT}
assert PROJECT_ROOT.is_dir(), f'Projeto ausente: {PROJECT_ROOT}'
source = PROJECT_ROOT / 'Dataset'
assert source.is_dir(), f'Dataset ausente: {source}'
shutil.copytree(source, DATA_ROOT / 'Dataset', dirs_exist_ok=True)
assert (DATA_ROOT/'Dataset/DFUC2022_train_images/100001.jpg').is_file()
print('Cache local pronto')
print('GPU:', subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True).stdout.strip())


## 2. Instalar a versão congelada do SAM

Usamos o repositório oficial fixado em um commit, e não uma versão móvel. O checkpoint ViT-B é baixado uma única vez para `models/sam` no Drive.


In [ ]:
SAM_COMMIT = "6fdee8f2727f4506cfbbe553e23b895e27956588"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/facebookresearch/segment-anything.git@{SAM_COMMIT}",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest"], check=True)
sam_dir = PROJECT_ROOT / "models/sam"
sam_dir.mkdir(parents=True, exist_ok=True)
sam_checkpoint = sam_dir / "sam_vit_b_01ec64.pth"
sam_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
if not sam_checkpoint.is_file():
    print("Baixando checkpoint oficial ViT-B...")
    urllib.request.urlretrieve(sam_url, sam_checkpoint)
print("SAM:", sam_checkpoint, f"{sam_checkpoint.stat().st_size/1024**2:.1f} MiB")


## 3. Conferir os arquivos congelados

A DR-IFT deve ser a execução da seed 42, melhor época 56 e threshold 0,18. Se algum item estiver ausente, pare aqui: não substitua por outra seed.


In [ ]:
CONFIG = PROJECT_ROOT / "configs/sam_vit_b_integration.yaml"
cfg = yaml.safe_load(CONFIG.read_text())
network_run = PROJECT_ROOT / cfg["network"]["run_directory"]
network_checkpoint = network_run / cfg["network"]["checkpoint_filename"]
required = [
    CONFIG,
    network_run / "config_used.yaml",
    network_checkpoint,
    sam_checkpoint,
    PROJECT_ROOT / cfg["data"]["manifest"],
]
for path in required:
    assert path.is_file(), f"Arquivo ausente: {path}"
print(
    "DR-IFT:",
    cfg["network"]["run_id"],
    "| época",
    cfg["network"]["checkpoint_epoch"],
    "| threshold",
    cfg["network"]["threshold"],
)
print("Todos os arquivos obrigatórios foram encontrados.")


## 4. Testar o contrato antes da GPU

Os testes verificam versão, threshold, bloqueio do teste, componentes múltiplos, máscara vazia e os argumentos entregues ao SAM.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_sam_refinement.py"],
    text=True,
    capture_output=True,
    env=env,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 5. Executar o smoke test

Para cada componente previsto, foram avaliadas três modalidades: caixa, `mask_input` e caixa com `mask_input`. O `mask_input` preserva o aspecto e o padding inferior/direito usados pelo SAM. A referência só é consultada depois da predição, para métricas e painel.


In [ ]:
command = [
    sys.executable,
    "scripts/run_sam_integration_smoke.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(CONFIG),
]
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
code = process.wait()
print(f"\nreturncode={code}")
if code:
    raise subprocess.CalledProcessError(code, command)


## 6. Ver o resultado

No mapa de erros: verde é acerto, laranja é falso positivo e vermelho é falso negativo. Os quatro exemplos são usados na verificação técnica. A seleção da modalidade usa a partição completa de validação.


In [ ]:
from IPython.display import display
from PIL import Image

runs = sorted((PROJECT_ROOT / "results/day12/smoke").glob("SAM1_ViTB_DRIFT_seed42_*"))
assert runs, "Nenhuma execução de integração técnica do SAM encontrada"
RUN = runs[-1]
report = json.loads((RUN / "technical_report.json").read_text())
print(json.dumps(report, indent=2, ensure_ascii=False))
display(Image.open(RUN / "sam_smoke_panel.png"))


## 7. Empacotar o que será analisado

O ZIP contém somente relatórios, métricas e painel do smoke test. Não precisa copiar o checkpoint SAM nem o checkpoint da rede.


In [ ]:
export_dir = PROJECT_ROOT / "exports/day12"
export_dir.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(export_dir / f"sam_integration_{RUN.name}"), "zip", RUN)
print("Arquivo exportado:", archive)


## Interpretação

O SAM recebeu apenas localização e logits produzidos pela DR-IFT. A referência foi usada somente para avaliação. A etapa seguinte aplica o procedimento às 294 imagens de validação.
